In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import torch
from transformers import pipeline



c:\Users\Atharv Jadhav\ML PROJECTS\COMPLETE-SUPPY-CHAIN\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
news_df = pd.DataFrame({
    "date": pd.to_datetime([
        "2024-01-10", "2024-01-15", "2024-02-02"
    ]),
    "headline": [
        "Heavy rainfall disrupts vegetable supply in Maharashtra",
        "Export restrictions raise onion prices",
        "Stable weather expected to improve crop yield"
    ],
    "content": [
        "Continuous rainfall has affected transportation and mandi arrivals.",
        "Government announced temporary export restrictions.",
        "Meteorological department predicts stable conditions."
    ],
    "source": ["News18", "Economic Times", "IMD"]
})


In [4]:
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 432.38it/s, Materializing param=pre_classifier.weight]                                  


In [5]:
def get_sentiment(text):
    result = sentiment_model(text[:512])[0]
    score = result['score']
    label = result['label']
    
    if label == "NEGATIVE":
        return -score
    else:
        return score

news_df['sentiment_score'] = news_df['headline'].apply(get_sentiment)
news_df[['headline', 'sentiment_score']]


,headline,sentiment_score
0,Heavy rainfall disrupts vegetable supply in Ma...,-0.999046
1,Export restrictions raise onion prices,-0.972101
2,Stable weather expected to improve crop yield,0.990939


In [6]:
RISK_KEYWORDS = {
    "weather": ["rain", "flood", "heatwave", "cyclone", "drought"],
    "logistics": ["transport", "port", "shipment", "supply"],
    "policy": ["ban", "export", "import", "restriction", "tax"],
    "conflict": ["strike", "war", "conflict", "protest"]
}

def detect_risk(text):
    text = text.lower()
    risk = 0
    
    for group, words in RISK_KEYWORDS.items():
        for w in words:
            if w in text:
                risk += 1
    return risk


In [7]:
news_df['risk_event_score'] = (
    news_df['headline'].apply(detect_risk) +
    news_df['content'].apply(detect_risk)
)


In [8]:
news_df['risk_index'] = (
    news_df['risk_event_score'] * 
    abs(news_df['sentiment_score'])
)


In [9]:
daily_risk = (
    news_df
    .groupby('date')
    .agg(
        avg_sentiment=('sentiment_score', 'mean'),
        total_events=('risk_event_score', 'sum'),
        risk_index=('risk_index', 'sum'),
        news_count=('headline', 'count')
    )
    .reset_index()
)

daily_risk.head()


,date,avg_sentiment,total_events,risk_index,news_count
0,2024-01-10,-0.999046,5,4.995231,1
1,2024-01-15,-0.972101,6,5.832608,1
2,2024-02-02,0.990939,0,0.000000,1
